In [ ]:
import pandas as pd
import numpy as np
from langdetect import detect, DetectorFactory
import argostranslate.package, argostranslate.translate
import re, os, json, time
import csv

In [ ]:
dados = pd.read_csv("./completoComLetras.csv")

In [ ]:
import csv, os
import pandas as pd
import re

_CTRL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f\u2028\u2029]")

def _limpa_texto(s, max_chars, proteger_formulas):
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""
    s = str(s)
    s = s.encode("utf-8", errors="replace").decode("utf-8")
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = _CTRL.sub("", s).strip()
    if max_chars and len(s) > max_chars:
        s = s[:max_chars]
    if proteger_formulas and s[:1] in ("=", "+", "-", "@"):
        s = " " + s
    return s

def saveArchiveB(data, caminho, sep=",", max_chars=32000,
                 proteger_formulas=True, salvar_parquet=False):
    out = data.copy()

    out = out.drop(columns=[c for c in out.columns
                            if c == "Unnamed: 0"
                            or str(c).endswith(("_x", "_y"))])

    cols_txt = out.select_dtypes(include=["object", "string", "str"]).columns
    truncadas = 0
    for c in cols_txt:
        antes = out[c].fillna("").astype(str).str.len()
        truncadas += int((antes > max_chars).sum()) if max_chars else 0
        out[c] = out[c].map(lambda s: _limpa_texto(s, max_chars, proteger_formulas))

    arquivo = caminho if caminho.lower().endswith(".csv") else caminho + ".csv"
    os.makedirs(os.path.dirname(arquivo) or ".", exist_ok=True)

    out.to_csv(
        arquivo,
        index=False,
        sep=sep,
        encoding="utf-8-sig",
        quoting=csv.QUOTE_NONNUMERIC,
        lineterminator="\r\n",
        # errors="replace"  ← REMOVIDO
    )

    if salvar_parquet:
        out.to_parquet(arquivo.replace(".csv", ".parquet"), index=False)

    chk = pd.read_csv(arquivo, sep=sep, keep_default_na=False,
                      dtype=str, encoding="utf-8-sig")
    ok_forma = chk.shape == out.shape
    ok_letra = (chk["letra"].reset_index(drop=True)
                == out["letra"].fillna("").astype(str).reset_index(drop=True)).all() \
               if "letra" in out.columns else True
    print(f"Salvo em {arquivo} | {out.shape[0]} linhas x {out.shape[1]} colunas")
    print(f"Forma preservada: {ok_forma} | Letras idênticas: {ok_letra} | Células truncadas: {truncadas}")
    return arquivo

In [ ]:
DetectorFactory.seed = 0   # resultado estável

def idioma(texto):
    try:
        return detect(texto[:2000])
    except Exception:
        return None

#unicas = dados.drop_duplicates("track_id")[["track_id", "letra"]].copy()
dados["idioma"] = dados["letra"].fillna("").map(idioma)
print(dados["idioma"].value_counts())

In [ ]:
# pip install transformers sentencepiece torch deep-translator
# ---------- configuração ----------
CACHE = "letras_en.jsonl"                 # só guarda traduções aprovadas
ORDEM = ["nllb", "argos" ,"google", "mymemory"]                # do pior ao melhor; cada motor só pega o que sobrou
NLLB_MODEL = "facebook/nllb-200-distilled-600M"
NLLB_BATCH = 16
GOOGLE_PAUSA = 1.0
GOOGLE_MAX = 4500  
MYMEMORY_LANGS = {
    "es": "es-ES",
    "pt": "pt-BR",
    "ja": "ja-JP",
    "ko": "ko-KR",
    "sw": "ja-JP",  # heurística: gíria lusófona mal detectada
    "tl": "ja-JP",
    "fr": "fr-FR",
}
MYMEMORY_MAX = 450  # limite seguro (MyMemory aceita ~500 chars)
MYMEMORY_PAUSA = 0.5

_argos = {}   # cache de tradutores já carregados por par de idioma

# Mapeamento do seu idioma detectado para os códigos ISO do Argos.
# 'pb' é o código para Português do Brasil; se não existir, cai em 'pt'.
ARGOS_LANGS = {
    "es": "es",
    "pt": "pb",   # ou "pt" se o pacote pb->en não estiver disponível
    "ja": "ja",
    "ko": "ko",
    "sw": "ja",   # misdetecção comum de pt/es
    "tl": "ja",
    "fr": "fr",
}
ARGOS_PAUSA = 0.1   # offline: pausa curta apenas para não travar o loop                       # limite do deep-translator é ~5000 por chamada

# idioma detectado -> (código NLLB, código Google)
LANGS = {
    "es": ("spa_Latn", "es"),
    "pt": ("por_Latn", "pt"),
    "ja": ("jpn_Jpan", "ja"),
    "ko": ("kor_Hang", "ko"),
}
# sw/tl/fr (raros) quase sempre são detecção errada de letras es/pt com gíria:
# não entram no NLLB; o Google tenta com detecção automática.
GOOGLE_AUTO = {"sw", "tl", "fr"}

# ---------- 0. base de letras únicas ----------
unicas = (dados.drop_duplicates("track_id")[["track_id", "idioma", "letra"]]
          .assign(letra=lambda d: d["letra"].fillna("").astype(str)))
unicas = unicas[unicas["letra"].str.strip() != ""]

# ---------- 1. cache ----------
def le_cache():
    if os.path.exists(CACHE) and os.path.getsize(CACHE) > 0:
        return pd.read_json(CACHE, lines=True)
    return pd.DataFrame(columns=["track_id", "letra_en", "motor"])

feitos = set(le_cache()["track_id"])

def grava(f, track_id, idioma, letra_en, motor):
    f.write(json.dumps({"track_id": track_id, "idioma": idioma,
                        "letra_en": letra_en, "motor": motor}, ensure_ascii=False) + "\n")
    f.flush()

# ---------- 2. checagem de qualidade ----------
_CJK = re.compile(r"[\u3040-\u30ff\u3400-\u9fff\uac00-\ud7af]")

def traducao_ok(orig, trad, idioma):
    if not trad or not trad.strip():
        return False
    lo = [l.strip().lower() for l in orig.split("\n") if l.strip()]
    lt = [l.strip().lower() for l in trad.split("\n") if l.strip()]
    if not lt:
        return False
    razao = len(trad) / max(len(orig), 1)
    if idioma in ("ja", "ko"):
        if len(_CJK.findall(trad)) / max(len(trad), 1) > 0.05:   # ainda em kana/hangul
            return False
        return 0.3 < razao < 6
    iguais = sum(a == b for a, b in zip(lo, lt)) / max(len(lo), 1)
    return iguais < 0.6 and 0.3 < razao < 3                      # não devolveu quase igual

# ---------- 3. motor NLLB (carrega só se for usado) ----------
_nllb = {}

def traduz_nllb(letra, idioma):
    import torch
    if idioma not in LANGS:
        return None
    if not _nllb:
        from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        _nllb["tok"] = AutoTokenizer.from_pretrained(NLLB_MODEL)
        #_nllb["mod"] = AutoModelForSeq2SeqLM.from_pretrained(NLLB_MODEL).to(dev).eval()
        _nllb["mod"] = AutoModelForSeq2SeqLM.from_pretrained(
                        NLLB_MODEL, torch_dtype=torch.float16 if dev == "cuda" else torch.float32
                ).to(dev).eval()
        _nllb["dev"] = dev
        print("NLLB carregado em", dev)
    tok, mod, dev = _nllb["tok"], _nllb["mod"], _nllb["dev"]
    tok.src_lang = LANGS[idioma][0]
    eng = tok.convert_tokens_to_ids("eng_Latn")

    linhas = letra.split("\n")
    unicas_l = list(dict.fromkeys(l for l in linhas if l.strip()))   # refrão repetido traduz 1 vez
    mapa = {}
    for i in range(0, len(unicas_l), NLLB_BATCH):
        lote = unicas_l[i:i + NLLB_BATCH]
        enc = tok(lote, return_tensors="pt", padding=True, truncation=True, max_length=200).to(dev)
        with torch.no_grad():
            out = mod.generate(**enc, forced_bos_token_id=eng, max_new_tokens=200, num_beams=4)
        for o, t in zip(lote, tok.batch_decode(out, skip_special_tokens=True)):
            mapa[o] = t
    return "\n".join(mapa.get(l, "") if l.strip() else "" for l in linhas)

# ---------- 4. motor Google (deep-translator) ----------
def traduz_google(letra, idioma):
    from deep_translator import GoogleTranslator
    src = "auto" if idioma in GOOGLE_AUTO or idioma not in LANGS else LANGS[idioma][1]
    gt = GoogleTranslator(source=src, target="en")

    linhas = letra.split("\n")
    blocos, atual, tam = [], [], 0
    for l in linhas:                                   # agrupa linhas sem passar do limite
        if tam + len(l) + 1 > GOOGLE_MAX and atual:
            blocos.append(atual); atual, tam = [], 0
        atual.append(l); tam += len(l) + 1
    if atual:
        blocos.append(atual)

    saida = []
    for b in blocos:
        texto = "\n".join(b)
        if not texto.strip():
            saida.append(texto); continue
        for tent in range(3):                          # retry com espera
            try:
                saida.append(gt.translate(texto) or "")
                break
            except Exception:
                time.sleep(3 * (tent + 1))
        else:
            return None                                # bloco falhou: faixa continua pendente
        time.sleep(GOOGLE_PAUSA)
    return "\n".join(saida)

def traduz_mymemory(letra, idioma):
    from deep_translator import MyMemoryTranslator
    src = MYMEMORY_LANGS.get(idioma)
    if src is None:
        return None
    mt = MyMemoryTranslator(source=src, target="en-GB")
    
    linhas = letra.split("\n")
    blocos, atual, tam = [], [], 0
    for l in linhas:
        if tam + len(l) + 1 > MYMEMORY_MAX and atual:
            blocos.append(atual); atual, tam = [], 0
        atual.append(l); tam += len(l) + 1
    if atual:
        blocos.append(atual)
    
    saida = []
    for b in blocos:
        texto = "\n".join(b)
        if not texto.strip():
            saida.append(texto); continue
        for tent in range(3):
            try:
                saida.append(mt.translate(texto) or "")
                break
            except Exception as e:
                time.sleep(2 * (tent + 1))
        else:
            return None
        time.sleep(MYMEMORY_PAUSA)
    return "\n".join(saida)
# ---------- 4c. motor Argos Translate (offline, open-source) ----------

def _argos_carrega(from_code, to_code="en"):
    """Baixa e instala o pacote de idioma na primeira vez, e retorna o tradutor."""
    import argostranslate.package
    import argostranslate.translate

    chave = (from_code, to_code)
    if chave in _argos:
        return _argos[chave]

    # Atualiza o índice de pacotes disponíveis e procura o par desejado
    argostranslate.package.update_package_index()
    disponiveis = argostranslate.package.get_available_packages()
    pacote = next(
        (p for p in disponiveis
         if p.from_code == from_code and p.to_code == to_code),
        None,
    )
    if pacote is None:
        return None   # par não disponível (ex.: pb->en pode não existir)

    # Instala se ainda não estiver instalado
    instalados = argostranslate.translate.get_installed_languages()
    from_lang = next((l for l in instalados if l.code == from_code), None)
    to_lang   = next((l for l in instalados if l.code == to_code), None)
    if from_lang is None or to_lang is None:
        caminho = pacote.download()
        argostranslate.package.install_from_path(caminho)
        instalados = argostranslate.translate.get_installed_languages()
        from_lang = next(l for l in instalados if l.code == from_code)
        to_lang   = next(l for l in instalados if l.code == to_code)

    tradutor = from_lang.get_translation(to_lang)
    _argos[chave] = tradutor
    print(f"Argos carregado: {from_code} -> {to_code}")
    return tradutor

def traduz_argos(letra, idioma):
    from_code = ARGOS_LANGS.get(idioma)
    if from_code is None:
        return None

    tradutor = _argos_carrega(from_code, "en")
    if tradutor is None:
        return None

    # O Argos já faz segmentação de sentenças internamente, mas dividir por
    # linhas ajuda a preservar a estrutura de versos da letra.
    linhas = letra.split("\n")
    saida = []
    for l in linhas:
        if not l.strip():
            saida.append("")
            continue
        for tent in range(3):
            try:
                saida.append(tradutor.translate(l) or "")
                break
            except Exception as e:
                print(f"  argos retry {tent+1}: {repr(e)[:80]}")
                time.sleep(1 * (tent + 1))
        else:
            return None   # linha falhou após 3 tentativas
        time.sleep(ARGOS_PAUSA)
    return "\n".join(saida)

MOTORES = {"nllb": traduz_nllb, "google": traduz_google, "mymemory": traduz_mymemory, "argos":traduz_argos,}

# ---------- 5. cascata ----------
# inglês não precisa de tradução: entra direto
with open(CACHE, "a", encoding="utf-8") as f:
    for r in unicas[(unicas["idioma"] == "en") & (~unicas["track_id"].isin(feitos))].itertuples():
        grava(f, r.track_id, "en", r.letra, "original")
        feitos.add(r.track_id)

for motor in ORDEM:
    pend = unicas[~unicas["track_id"].isin(feitos)]
    print(f"\n[{motor}] pendentes: {len(pend)}")
    if pend.empty:
        break
    ok = 0
    with open(CACHE, "a", encoding="utf-8") as f:
        for r in pend.itertuples():
            try:
                trad = MOTORES[motor](r.letra, r.idioma)
            except Exception as e:
                print("erro", motor, r.track_id, repr(e)[:100])
                continue
            if trad is not None and traducao_ok(r.letra, trad, r.idioma):
                grava(f, r.track_id, r.idioma, trad, motor)
                feitos.add(r.track_id)
                ok += 1
    print(f"[{motor}] aprovadas: {ok}")

# ---------- 6. merge no dados ----------
en = le_cache().drop_duplicates("track_id")[["track_id", "letra_en", "motor"]] \
               .rename(columns={"motor": "motor_traducao"})
dados = dados.drop(columns=[c for c in ["letra_en", "motor_traducao"] if c in dados.columns]) \
             .merge(en, on="track_id", how="left")

# ---------- 7. relatório ----------
print("\nTraduções por motor (tracks únicas):")
print(dados.drop_duplicates("track_id")["motor_traducao"].value_counts(dropna=False))

sem = dados[dados["letra_en"].isna() & (dados["letra"].fillna("").str.strip() != "")] \
        .drop_duplicates("track_id")
print(len(sem), "tracks sem tradução aprovada")
display(sem[["track_id", "idioma", "title", "artists"]])

In [ ]:
dados.columns

In [ ]:
saveArchiveB(dados, "finalData")